In [44]:

# Uppdaterar ändringar i src utan att behöva starta om kernel:

%load_ext autoreload 
%autoreload 2
%matplotlib inline


import sys
import os
import time

# Check kernel
print(f'path: {sys.executable}')

from typing import Tuple
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

try:
    import lets_be_rational.LetsBeRational as lbr
    print("Loaded via package import")

except ModuleNotFoundError:
    print("Package import failed. Loading .so manually...")

    import importlib.util
    
    so_path = Path("../../venv/lib/python3.13/site-packages/lets_be_rational/_LetsBeRational.cpython-313-darwin.so")

    if not so_path.exists():
        raise FileNotFoundError(f"Could not find shared library at {so_path}")

    spec = importlib.util.spec_from_file_location("_LetsBeRational", so_path)
    lbr = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(lbr)

    print("Loaded via direct .so import")

print("Final module:", lbr)

from heston_project.models.DDN import *
from heston_project.utils import *


# Device Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Reproducibility 
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if device.type == 'cuda':
    torch.cuda.manual_seed(SEED)

print("Setup Complete.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
path: /Users/manswestman/Kandidatarbete/venv/bin/python
Package import failed. Loading .so manually...
Loaded via direct .so import
Final module: <module '_LetsBeRational' from '/Users/manswestman/Kandidatarbete/notebooks/2_model_train/../../venv/lib/python3.13/site-packages/lets_be_rational/_LetsBeRational.cpython-313-darwin.so'>
Using device: cpu
Setup Complete.


In [47]:
df = pd.read_csv('../../data/full_dataset_training_200000(2).csv')

# Shuffle with a fixed random state for reproducibility
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df_scaler = df.copy()

#Scale theta & v0 (Ska dessa behållas i vanlig form också för att inte scalea om i output)
df_scaler[['theta', 'v0']] = np.sqrt(df_scaler[['theta', 'v0']])
sqrt_copy = df_scaler[['theta', 'v0']].copy()

inputs = df_scaler[['kappa', 'theta', 'v0', 'rho', 'sigma', 'r', 'lm', 'T']]
labels = df_scaler[['price', 'IV', 'vega', 'dtheta', 'dsigma', 'drho', 'dkappa', 'dv0']]

# See utils for param bounds

# param_bounds = {
#     "lm":    [-1.5, 1.5],
#     "r":     [-0.01, 0.10],
#     "T":     [1/52, 3.0],
#     "theta": [0.0, 1.0],
#     "sigma": [0.1, 1.5],
#     "rho":   [-0.95, 0.0],
#     "kappa": [0.05, 5.0],
#     "v0":    [0.0, 1.0],
# }


# Skalar inputs till [-1,1]
inputs = scale_inputs(inputs, zero_centered=True)

print(len(inputs), "rows total before filtering")
# Rensa data från NaN och små värden
vega_floor = 1e-4 # Så att vi inte får extrema IV greeks (-> infinity)

good = ( # också en boolean mask
    np.isfinite(labels['IV']) &
    np.isfinite(labels['vega']) & (labels['vega'] > vega_floor)
)
inputs = inputs.loc[good].reset_index(drop=True)
labels = labels.loc[good].reset_index(drop=True)
sqrt_copy = sqrt_copy.loc[good].reset_index(drop=True)

# Skalar greek targets till dP_tilde/dx (där x=inputs skalat till [-1,1]) genom kedjeregeln
def scale_greek_target(diff, xmin, xmax):
    return ((xmax - xmin) * diff) / 2

# Konverterar price greeks -> scaled input price greeks -> scaled input IV greeks
greek_max_mins = {}

for k, (xmin, xmax) in param_bounds.items():
    greek_label = 'd' + str(k)

    if k in ["theta", "v0"] and greek_label in labels.columns: #En extra term i kedjeregeln krävs för att ta hänsyn till sqrt-transformationen
        labels[greek_label] = scale_greek_target(labels[greek_label], xmin, xmax) * (2 * sqrt_copy[k])
    elif greek_label in labels.columns:
        labels[greek_label] = scale_greek_target(labels[greek_label], xmin, xmax)

    if greek_label in labels.columns:
        labels[greek_label] = labels[greek_label] / labels['vega'] # IV greeks

        greek_max_mins[greek_label] = [labels[greek_label].max()]
        greek_max_mins[greek_label].append(labels[greek_label].min())

# labels = labels.drop(columns=['price', 'vega'])

#Split skalat data
X_train, X_val, y_train, y_val = train_test_split(inputs, labels, test_size=0.1, random_state=42)
print(f"vega min BEFORE greek-percentile filter (= vega_floor): {y_train['vega'].min():.6e}")

def filter_greek_outliers(X, y, greek_cols, thresholds=None, q=99):
    """Drop rows where |any greek| exceeds the q-th percentile of |that greek|.
    Pass `thresholds` to reuse train-set cutoffs on val (no leakage)."""
    if thresholds is None:
        thresholds = {c: np.percentile(np.abs(y[c]), q) for c in greek_cols}
    mask = np.ones(len(y), dtype=bool)
    for c in greek_cols:
        mask &= (np.abs(y[c]).values <= thresholds[c])
    return X.loc[mask].reset_index(drop=True), y.loc[mask].reset_index(drop=True), thresholds, mask

greek_cols = ['dtheta', 'dsigma', 'drho', 'dkappa', 'dv0']

y_train_pre = y_train.copy()
X_train, y_train, thr, m_tr = filter_greek_outliers(X_train, y_train, greek_cols, q=99)
X_val,   y_val,   _,   m_va = filter_greek_outliers(X_val,   y_val,   greek_cols, thresholds=thr)

print(f"vega min AFTER  greek-percentile filter (effective vega bound): {y_train['vega'].min():.6e}")


print(f"Train kept: {m_tr.sum()}/{len(m_tr)} ({100*m_tr.mean():.2f}%)")
print(f"Val   kept: {m_va.sum()}/{len(m_va)} ({100*m_va.mean():.2f}%)")

dropped = y_train_pre.loc[~m_tr, ['vega'] + greek_cols].copy()
dropped['max_abs_greek'] = dropped[greek_cols].abs().max(axis=1)
print(f"\nDropped rows: {len(dropped)}")
print(f"vega stats among dropped  -> min: {dropped['vega'].min():.3e}, median: {dropped['vega'].median():.3e}, max: {dropped['vega'].max():.3e}")
print("\nSmallest-vega dropped rows:")
print(dropped.nsmallest(5, 'vega'))
print("\nLargest-|greek| dropped rows:")
print(dropped.nlargest(5, 'max_abs_greek'))
print("\nPer-greek 99th-percentile thresholds (|greek|):")
for c in greek_cols: print(f"  {c}: {thr[c]:.3e}")

print(min(y_train['IV']))
print(np.percentile(y_train['IV'],99.5))
print(min(y_val['IV']))
print(np.percentile(y_val['IV'],99.5))


print(len(X_train) + len(X_val), "rows total after filtering")


# # Rensa data från NaN och små värden
# vega_floor = 1e-4 # Så att vi inte får extrema IV greeks (-> infinity)

# good = ( # också en boolean mask
#     labels['vega'] > vega_floor
# )
# X_train = X_train.loc[good].reset_index(drop=True)
# y_train = y_train.loc[good].reset_index(drop=True)

train_dataset = TensorDataset(
    torch.tensor(X_train.values, dtype=torch.float32),
    torch.tensor(y_train.values, dtype=torch.float32)
    )
val_dataset = TensorDataset(
    torch.tensor(X_val.values, dtype=torch.float32),
    torch.tensor(y_val.values, dtype=torch.float32)
    )

train_loader = DataLoader(train_dataset, batch_size=256,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=2048)


199820 rows total before filtering
vega min BEFORE greek-percentile filter (= vega_floor): 1.000215e-04
vega min AFTER  greek-percentile filter (effective vega bound): 1.000215e-04
Train kept: 164766/172622 (95.45%)
Val   kept: 18302/19181 (95.42%)

Dropped rows: 7856
vega stats among dropped  -> min: 1.004e-04, median: 1.764e-01, max: 6.897e-01

Smallest-vega dropped rows:
            vega    dtheta    dsigma      drho    dkappa       dv0  \
110601  0.000100  0.004254 -0.141323  0.259054 -0.039951  0.558384   
118579  0.000101  0.040141 -0.066330  0.302253  0.069330  0.423641   
89355   0.000101  0.004664 -0.032952  0.486117 -0.006701  0.343119   
163234  0.000102  0.027741 -0.003761  0.473865 -0.065083  0.067255   
64350   0.000102  0.328168 -0.102568  0.516796  0.092840  0.183961   

        max_abs_greek  
110601       0.558384  
118579       0.423641  
89355        0.486117  
163234       0.473865  
64350        0.516796  

Largest-|greek| dropped rows:
            vega    dtheta 

**Weights for Greek Loss**


In [31]:
greek_w = {}
raw_weights = []
keys_in_order = []

# 1. Iterate through greeks
for g in [f'd{k}' for k in param_bounds if f'd{k}' in labels.columns]:
    print(f"{g}: {np.max(y_train[g].to_numpy())}")
    
    # Calculate range
    greek = y_train[g].to_numpy()
    
    lo, hi = np.quantile(greek, [0.01, 0.99]) # ger ett lite mer robust intervall (ignorerar de mest extrema 1% på varje sida)
    print(hi)
    val_range = hi - lo
    
    # Calculate raw weight
    raw_weight = 1.0 / (val_range**2 + 1e-8)
    
    raw_weights.append(raw_weight)
    keys_in_order.append(g)

# 2. Convert the Python list to a PyTorch tensor BEFORE doing math
raw_weights_tensor = torch.tensor(raw_weights, dtype=torch.float32)

# 3. Normalize the tensor
normalized_weights_tensor = raw_weights_tensor / torch.mean(raw_weights_tensor)

# 4. Map them back to the dictionary for easy lookup
for i, k in enumerate(keys_in_order):
    # .item() extracts the standard Python float from the 0D tensor
    greek_w[f'{k}'] = normalized_weights_tensor[i].item() 

print(greek_w)


dtheta: 0.4719760501390359
0.46596917088795253
dsigma: 0.19496610752683768
0.12080414140697351
drho: 0.26043082322595806
0.20392099983266054
dkappa: 0.7391615657952905
0.5123239326068257
dv0: 0.47873846446196033
0.45680945097910647
{'dtheta': 0.6531310677528381, 'dsigma': 1.790850043296814, 'drho': 1.6973240375518799, 'dkappa': 0.17034976184368134, 'dv0': 0.6883451342582703}


Ska vi testa denna????

In [38]:
greek_w = {}
raw_weights = []
keys_in_order = []

# 1. Iterate through greeks
for g in [f'd{k}' for k in param_bounds if f'd{k}' in labels.columns]:
    
    col = y_train[g]
    print(y_train[g].var())

    # Compute percentile bounds
    lower = col.quantile(0.01)
    upper = col.quantile(0.99)

    # Filter to 1st–99th percentile
    col_filtered = col[(col >= lower) & (col <= upper)]

    # Compute variance on filtered data

    col_variance = col_filtered.var()

    print(col_variance)
    
    raw_weight = 1.0 / (col_variance + 1e-8)
        
    raw_weights.append(raw_weight)
    keys_in_order.append(g)

# 2. Convert the Python list to a PyTorch tensor BEFORE doing math
raw_weights_tensor = torch.tensor(raw_weights, dtype=torch.float32)

# 3. Normalize the tensor
normalized_weights_tensor = raw_weights_tensor / torch.mean(raw_weights_tensor)

# 4. Map them back to the dictionary for easy lookup
for i, k in enumerate(keys_in_order):
    # .item() extracts the standard Python float from the 0D tensor
    greek_w[f'{k}'] = normalized_weights_tensor[i].item() 

print(greek_w)


0.018779825656886767
0.017951447655486865
0.003261107344682132
0.0024492358471834416
0.004459774387797551
0.0031665246329720008
0.09266659608459765
0.016414496731430545
0.01448943485403477
0.013373661548205932
{'dtheta': 0.30423930287361145, 'dsigma': 2.2298858165740967, 'drho': 1.7247682809829712, 'dkappa': 0.3327263295650482, 'dv0': 0.4083798825740814}


**Training & Validation Steps**

In [32]:
# Bestämmer vilka greeks vi plockar ut för pred_grads och y_grads som ska användas i lossfunktionen.
relevant_greeks = ['kappa', 'v0', 'rho', 'sigma', 'theta']

# y och full_grad bygger på dataset med icke-matchande kolumnordning     
input_columns = list(inputs.columns)
output_columns = list(labels.columns)

index_in_input = torch.as_tensor([input_columns.index(k) for k in relevant_greeks], device=device)
index_in_output = torch.as_tensor([output_columns.index(f'd{k}') for k in relevant_greeks], device=device)

weights_list = [greek_w[f'd{k}'] for k in relevant_greeks]
greek_weights_tensor = torch.tensor(weights_list, dtype=torch.float32, device=device)
# vega_index = torch.as_tensor([output_columns.index('vega')], device=device)

In [33]:
def calculate_loss(pred_IV, y_batch, pred_grads, index_in_output, greek_weights_tensor, lambda_IV, lambda_g):
    # OBS! (Index 0 after dropping 'price' and 'vega')
    y_IV = y_batch[:, 0:1]
    
    # IV Loss (Standard MSE)
    l_IV = torch.mean((pred_IV - y_IV) ** 2)
    
    # 2. Greeks Target
    y_grads = y_batch[:, index_in_output]
    
    # Greek Loss: Calculate MSE for each Greek independently, then apply weights
    mse_per_greek = torch.mean((pred_grads - y_grads) ** 2, dim=0)
    l_g = torch.sum(mse_per_greek * greek_weights_tensor)
    
    # 3. Total Combined Loss
    loss = (lambda_IV * l_IV) + (lambda_g * l_g)
    
    return loss, l_IV, l_g

In [34]:
def train_step(model, criterion, batch: tuple, optimizer, weights):

        x, y = batch

        x.requires_grad_(True)

        optimizer.zero_grad(set_to_none=True)

        # IV loss
        pred_IV = model(x)
        y_IV = y[:, 1:2]
        l_IV = criterion(pred_IV, y_IV)


        # Compute gradients (Greeks)
        full_grads = torch.autograd.grad(
            outputs=pred_IV,
            inputs=x,
            grad_outputs=torch.ones_like(pred_IV),
            create_graph=True, # OBS detta är inte för att få greeks utan för att beräkna gradienten av greeksen för optimeraren!!
            # retain_graph=True, är sant automatiskt
            only_inputs=True
        )[0]

        # Greek loss
        pred_grads = full_grads[:, index_in_input]
                
        loss, l_IV, l_g = calculate_loss(
            pred_IV = pred_IV, 
            y_batch = y, 
            pred_grads = pred_grads, 
            index_in_output = index_in_output, 
            greek_weights_tensor = greek_weights_tensor,
            lambda_IV = weights['lambda_IV'], 
            lambda_g = weights['lambda_g']
        )

        # Compute loss gradients and update parameters
        loss.backward()
        optimizer.step()

        return {"loss": loss.item(), "l_IV": l_IV.item(), "l_g": l_g.item()}

In [35]:
def val_step(model, criterion, batch: Tuple, weights):

    x, y = batch

    x.requires_grad_(True)

    # IV loss
    pred_IV = model(x)    

    # Compute gradients (Greeks)
    ones = torch.ones_like(pred_IV)
    full_grads = torch.autograd.grad(
            outputs=pred_IV,
            inputs=x,
            grad_outputs=ones,
            create_graph=False, # Behövs ej vid test/validering
            retain_graph=False
        )[0]
        

    # Greek loss
    pred_grads = full_grads[:, index_in_input]
                    
    loss, l_IV, l_g = calculate_loss(
            pred_IV = pred_IV, 
            y_batch = y, 
            pred_grads = pred_grads, 
            index_in_output = index_in_output, 
            greek_weights_tensor = greek_weights_tensor,
            lambda_IV = weights['lambda_IV'], 
            lambda_g = weights['lambda_g']
        )
    
    return {"loss": loss.item(), "l_IV": l_IV.item(), "l_g": l_g.item()}

**HYPERPARAMETERS**

In [36]:
# Loss weights
lambda_IV = 10
lambda_g = 1

# Model parameters 
nr_input_dim = 8
nr_hidden_layers = 7
nr_neurons = 250

# Optimizer settings
learning_rate = 3e-4
weight_decay = 1e-5
scheduler_patience = 7
scheduler_factor = 0.5

# Training loop
EPOCHS = 450
patience_early_stopping = 450

In [37]:
# initialisera modellen till device
model = DDN(input_dim=nr_input_dim, hidden_layers=nr_hidden_layers, neurons=nr_neurons).to(device)

# Loss + Optimizer
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
# Överväg att sänka lr och ska vi ha L-BFGS???
#scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=8, factor=0.5) # bestämer hur och när vi ska ändra lr 
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=150, T_mult=1, eta_min=1e-6)

**TRAINING LOOP**

In [38]:

# Path for saving best model state
import heston_project
PKG_DIR = Path(heston_project.__file__).resolve().parent
SAVE_DIR = PKG_DIR / "models" / "saved"
out_path = SAVE_DIR / "DDN_IV_removed_10p.pth"

# Tracker for early stopping
counter = 0

# Loss dicts
train_losses = {'total_train_loss': [], 'lp_train': [], 'lg_train': []}
val_losses = {'total_val_loss': [], 'lp_val': [], 'lg_val': []}

best_val_loss = 1000

# Loss weights
weights_dict = {'lambda_IV': lambda_IV, 'lambda_g': lambda_g}

print(f'Starting Training on {device}...')
start_time = time.time()        ##########


for epoch in range(EPOCHS):
    model.train()
    running_tot_loss, running_lp_loss, running_lg_loss = 0.0, 0.0, 0.0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        losses = train_step(model, criterion, batch = (batch_X, batch_y), optimizer = optimizer, weights = weights_dict)

        running_tot_loss += losses['loss']
        running_lp_loss += losses['l_IV']
        running_lg_loss += losses['l_g']
    
    avg_train_total_loss = running_tot_loss/len(train_loader)
    avg_train_lp_loss = running_lp_loss/len(train_loader)
    avg_train_lg_loss = running_lg_loss/len(train_loader)

    train_losses['total_train_loss'].append(avg_train_total_loss)
    train_losses['lp_train'].append(avg_train_lp_loss)
    train_losses['lg_train'].append(avg_train_lg_loss)

    model.eval()
    running_tot_loss, running_lp_loss, running_lg_loss = 0.0, 0.0, 0.0

    # OBS ANVÄND ABSOLUT INTE TORCH NO GRAD
    for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)

            losses = val_step(model, criterion = criterion, batch = (batch_X, batch_y), weights = weights_dict)

            running_tot_loss += losses['loss']
            running_lp_loss += losses['l_IV']
            running_lg_loss += losses['l_g']
    
    avg_val_total_loss = running_tot_loss/len(val_loader)
    avg_val_lp_loss = running_lp_loss/len(val_loader)
    avg_val_lg_loss = running_lg_loss/len(val_loader)


    val_losses['total_val_loss'].append(avg_val_total_loss)
    val_losses['lp_val'].append(avg_val_lp_loss)
    val_losses['lg_val'].append(avg_val_lg_loss)

    # Uppdatera Learning Rate
    scheduler.step()

    if avg_val_total_loss < best_val_loss:
        best_val_loss = avg_val_total_loss
        best_loss_price = avg_val_lp_loss
        best_loss_greeks = avg_val_lg_loss
        torch.save(model.state_dict(), out_path) # Save the best version
        counter = 0 # Återställ!!!
    else:
        counter += 1
        if counter >= patience_early_stopping:
            print(f"Early stopping triggered at epoch {epoch}")
            break
    
    if (epoch + 1) % 5 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch [{epoch+1:3d}/{EPOCHS}] | LR: {current_lr:.2e}")
        print(f"  Train -> Tot: {avg_train_total_loss:.4e} | IV: {avg_train_lp_loss:.4e} | Greeks: {avg_train_lg_loss:.4e}")
        print(f"  Val   -> Tot: {avg_val_total_loss:.4e} | IV: {avg_val_lp_loss:.4e} | Greeks: {avg_val_lg_loss:.4e}")
        print("-" * 70)

print("*" * 80)
print(f"Best Val Loss -> Tot: {best_val_loss:.4e} | IV: {best_loss_price:.4e} | Greeks: {best_loss_greeks:.4e}")
print("*" * 80)

end_time = time.time()
print(f"Training Finished. Total Time: {end_time - start_time:.2f} seconds")

loss_df = pd.DataFrame({
    "epoch": range(1, len(train_losses["total_train_loss"]) + 1),
    "train_total_loss": train_losses["total_train_loss"],
    "train_lp_loss": train_losses["lp_train"],
    "train_lg_loss": train_losses["lg_train"],
    "val_total_loss": val_losses["total_val_loss"],
    "val_lp_loss": val_losses["lp_val"],
    "val_lg_loss": val_losses["lg_val"],
})

loss_df.to_csv("losses/loss_data/DDN_IV_loss_history_removed_10p.csv", index=False)

# old res:  IV: 1.2292e-06 | Greeks: 5.2663e-05


Starting Training on cpu...
Epoch [  5/450] | LR: 2.99e-04
  Train -> Tot: 2.8590e-04 | IV: 1.5687e-05 | Greeks: 1.2903e-04
  Val   -> Tot: 3.0446e-04 | IV: 1.8778e-05 | Greeks: 1.1668e-04
----------------------------------------------------------------------
Epoch [ 10/450] | LR: 2.97e-04
  Train -> Tot: 1.7223e-04 | IV: 1.0689e-05 | Greeks: 6.5342e-05
  Val   -> Tot: 1.0912e-04 | IV: 6.1342e-06 | Greeks: 4.7776e-05
----------------------------------------------------------------------
Epoch [ 15/450] | LR: 2.93e-04
  Train -> Tot: 1.1031e-04 | IV: 6.8653e-06 | Greeks: 4.1656e-05
  Val   -> Tot: 6.2519e-05 | IV: 3.2180e-06 | Greeks: 3.0339e-05
----------------------------------------------------------------------
Epoch [ 20/450] | LR: 2.87e-04
  Train -> Tot: 8.8601e-05 | IV: 5.6151e-06 | Greeks: 3.2450e-05
  Val   -> Tot: 8.9354e-05 | IV: 5.3711e-06 | Greeks: 3.5642e-05
----------------------------------------------------------------------
Epoch [ 25/450] | LR: 2.80e-04
  Train -> To